In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
!pip install datasets
from datasets import load_dataset

# ================ Load Dataset ===================
dataset = load_dataset("dair-ai/emotion")

# Split dataset
train_texts = list(dataset["train"]["text"])
train_labels = list(dataset["train"]["label"])
test_texts = list(dataset["test"]["text"])
test_labels = list(dataset["test"]["label"])

# ================ Tokenization ===================
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(texts):
    return tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

train_encodings = tokenize(train_texts)
test_encodings = tokenize(test_texts)

# ================ Custom Dataset ===================
class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}, self.labels[idx]

train_dataset = EmotionDataset(train_encodings, train_labels)
test_dataset = EmotionDataset(test_encodings, test_labels)

# ================ Dataloader ===================
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

# ================ Model Definition ===================
class EmotionClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes):
        super(EmotionClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.conv1d = nn.Conv1d(in_channels=embedding_dim, out_channels=64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.embedding(x)        # (batch_size, seq_len, embedding_dim)
        x = x.permute(0, 2, 1)       # (batch_size, embedding_dim, seq_len)
        x = self.conv1d(x)           # (batch_size, out_channels, seq_len)
        x = self.relu(x)
        x = self.global_avg_pool(x)  # (batch_size, out_channels, 1)
        x = x.squeeze(2)             # (batch_size, out_channels)
        x = self.fc(x)               # (batch_size, num_classes)
        return x

# Model instance
vocab_size = tokenizer.vocab_size
embedding_dim = 128  # You can choose the embedding dimension
num_classes = len(set(train_labels))
model = EmotionClassifier(vocab_size, embedding_dim, num_classes)

# ================ Training ===================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch, labels in train_loader:
        batch = {key: val.to(device) for key, val in batch.items()}
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(batch["input_ids"])
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss / len(train_loader):.4f}")

# ================ Evaluation ===================
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch, labels in test_loader:
        batch = {key: val.to(device) for key, val in batch.items()}
        labels = labels.to(device)

        outputs = model(batch["input_ids"])
        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total * 100
print(f"Test Accuracy: {accuracy:.2f}%")

# ================ Prediction ===================
def predict(sentence):
    model.eval()
    encoded = tokenize([sentence])
    encoded = {key: val.to(device) for key, val in encoded.items()}

    with torch.no_grad():
        output = model(encoded["input_ids"])
        prediction = torch.argmax(output, dim=1).cpu().item()

    emotions = ["sadness", "joy", "love", "anger", "fear", "surprise"]
    return emotions[prediction]

sample_sentence = "I’m so excited about this new adventure!"
print(f"Predicted Emotion: {predict(sample_sentence)}")

Epoch [1/5], Loss: 1.4792
Epoch [2/5], Loss: 0.8629
Epoch [3/5], Loss: 0.4256
Epoch [4/5], Loss: 0.2499
Epoch [5/5], Loss: 0.1655
Test Accuracy: 87.45%
Predicted Emotion: joy


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
!pip install datasets
from datasets import load_dataset

In [ ]:
# 1. load dataset
dataset= load_dataset("dair-ai/emotion")
train_texts = list(dataset["train"]["text"])
train_labels = list(dataset["train"]["label"])
test_texts = list(dataset["test"]["text"])
test_labels = list(dataset["test"]["label"])

# 2. Tokenization
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
def tokenize(texts):
  return tokenizer(texts, padding=True, truncation=True, return_tensors='pt')

train_encodings = tokenize(train_texts)
test_encodings = tokenize(test_texts)


# 3. Custom dataset
class EmotionDataset(Dataset):
  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = torch.tensor(labels, dtype=torch.int64)

  def __len__(self):
    return len(self.labels)
  def __getitem__(self, idx):
    item = {key: val[idx] for key, val in self.encodings.items()}
    return item, self.labels[idx]


train_dataset = EmotionDataset(train_encodings, train_labels)
test_dataset = EmotionDataset(test_encodings, test_labels)


In [ ]:
# 4. Dataloader
train_loader = DataLoader(train_dataset, batch_size = 16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size = 16)

# 5. Modeling
class EmotionClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes):
      super().__init__()
      self.embedding = nn.Embedding(vocab_size, embedding_dim) # output: (N, L, C) -> (64, 87, 128)
      self.conv1d = nn.Conv1d(in_channels=embedding_dim, out_channels=64, kernel_size=3, padding=1) # 1 ảnh trong batch có shape output (L_out, C_out) -> (S', 64) còn L_in = emb_dim = 128
      self.relu = nn.ReLU()
      self.global_avg_pool = nn.AdaptiveAvgPool1d(1) # 1
      self.dropout = nn.Dropout(0.5)
      self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
      x = self.embedding(x) #(N, seq_len, emb_dim)
      x = x.permute(0,2,1) # (N, emb_dim, seq_len)
      x = self.conv1d(x) # (batch_size, out_channels, seq_len)
      x = self.relu(x)
      x = self.global_avg_pool(x) # (batch_size, out_channels, 1)
      x = x.squeeze(2) # (batch_size, out_channels)
      x = self.dropout(x)
      x = self.fc(x) # (batch_size, num_classes)
      return x

# model instance
vocab_size = tokenizer.vocab_size
embedding_dim = 128
num_classes = len(set(train_labels))
model = EmotionClassifier(vocab_size, embedding_dim, num_classes)
model


EmotionClassifier(
  (embedding): Embedding(30522, 128)
  (conv1d): Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu): ReLU()
  (global_avg_pool): AdaptiveAvgPool1d(output_size=1)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=64, out_features=6, bias=True)
)

In [ ]:
# training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
num_epochs=20
for epoch in range(num_epochs):
  model.train()
  total_loss = 0
  for batch, labels in train_loader:
    batch = {key:val.to(device) for key, val in batch.items()}
    labels = labels.to(device)

    optimizer.zero_grad()
    outputs = model(batch['input_ids'])
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  print(f"Epochs [{epoch+1}/{num_epochs}], Loss = {total_loss/len(train_loader):.4f}")
model.eval()
correct = 0
total = 0

with torch.no_grad():
  for batch, labels in test_loader:
    batch = {key:val.to(device) for key, val in  batch.items()}
    labels = labels.to(device)
    outputs = model(batch['input_ids'])
    predictions = torch.argmax(outputs, dim=1)
    correct += (predictions==labels).sum().item()
    total += labels.size(0)

accuracy = correct / total * 100
print (f"Test accuracy: {accuracy:.2f}%")









Epochs [1/20], Loss = 0.0893
Epochs [2/20], Loss = 0.0813
Epochs [3/20], Loss = 0.0761
Epochs [4/20], Loss = 0.0723
Epochs [5/20], Loss = 0.0689
Epochs [6/20], Loss = 0.0625
Epochs [7/20], Loss = 0.0586
Epochs [8/20], Loss = 0.0545
Epochs [9/20], Loss = 0.0580
Epochs [10/20], Loss = 0.0545
Epochs [11/20], Loss = 0.0507
Epochs [12/20], Loss = 0.0506
Epochs [13/20], Loss = 0.0470
Epochs [14/20], Loss = 0.0463
Epochs [15/20], Loss = 0.0401
Epochs [16/20], Loss = 0.0404
Epochs [17/20], Loss = 0.0388
Epochs [18/20], Loss = 0.0392
Epochs [19/20], Loss = 0.0414
Epochs [20/20], Loss = 0.0381
Test accuracy: 88.50%


In [ ]:
# evaluation
def predict (sentence):
  model.eval()
  encoded = tokenize([sentence])
  encoded = {key:val.to(device) for key, val in encoded.items()}

  with torch.no_grad():
    output = model(encoded['input_ids'])
    prediction = torch.argmax(output, dim=1).cpu().item()
  emotions = ["sadness", "joy", "love", "anger", "fear", "surprise"]
  return emotions[prediction]
sample_sentence = "I’m so excited about this new adventure!"

predict(sample_sentence)

'joy'

# Ghi chú và kiểm tra

In [ ]:
train_encodings['input_ids'].shape # 16000: là tổng số câu trong bộ dataset (có thể check bằng len(train_texts)) | 87 là max seq len, được tokenizer lấy theo seq len của câu dài nhất

torch.Size([16000, 87])

In [ ]:
next(iter(train_loader))[0]['input_ids'].shape # shape=64, đúng bằng batch_size -> oke data loader đúng rồi

torch.Size([64, 87])

In [ ]:
embedding = nn.Embedding(tokenizer.vocab_size, embedding_dim= 128)
batch, labels = next(iter(train_loader))
emb_out = embedding(batch['input_ids'])
print("Shape:", emb_out.shape)          # torch.Size([64, 87, 128]) -> 87 là số token trong một chuỗi, 128 là số chiều mô tả cho 1 token
print("Vector token đầu tiên (5 số đầu):", emb_out[0, 0, :5]) # 5 chiều/128 chiều của token đầu của câu đầu




Shape: torch.Size([64, 87, 128])
Vector token đầu tiên (5 số đầu): tensor([-2.3961, -1.1159,  0.9772,  1.0533, -0.2892], grad_fn=<SliceBackward0>)


In [ ]:
import numpy as np
a = np.array([[0,1,2,3],
              [4,5,6,7]])
b = np.array([[12,13,14,15],
              [8,9,10,11]])

print(f'A shape: {a.shape}')
print(f'B shape: {b.shape}')
print(np.sum((a, b), axis = 0))
print(np.sum((a, b), axis = 0).shape)
print(np.sum((a, b), axis = 1))
print(np.sum((a, b), axis = 1).shape)





A shape: (2, 4)
B shape: (2, 4)
[[12 14 16 18]
 [12 14 16 18]]
(2, 4)
[[ 4  6  8 10]
 [20 22 24 26]]
(2, 4)


In [ ]:
import numpy as np
a = np.array([[0,1,2,3],
              [4,5,6,7]])

print(f'A shape: {a.shape}')
print(np.sum((a), axis = 0))
print(np.sum((a), axis = 0).shape)
print(np.sum((a), axis = 1))
print(np.sum((a), axis = 1).shape)

A shape: (2, 4)
[ 4  6  8 10]
(4,)
[ 6 22]
(2,)


In [ ]:
dict({'mot': 1, 'hai':2}).keys()

dict_keys(['mot', 'hai'])

In [ ]:
item = {key: val[idx] for key, val in self.encodings.items()}

'mot'

In [ ]:
train_encodings.items()

ItemsView({'input_ids': tensor([[  101,  1045,  2134,  ...,     0,     0,     0],
        [  101,  1045,  2064,  ...,     0,     0,     0],
        [  101, 10047,  9775,  ...,     0,     0,     0],
        ...,
        [  101,  1045,  2514,  ...,     0,     0,     0],
        [  101,  1045,  2514,  ...,     0,     0,     0],
        [  101,  1045,  2113,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])})

In [ ]:
# ================ Model Definition ===================
class EmotionClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes):
        super(EmotionClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.conv1d = nn.Conv1d(in_channels=embedding_dim, out_channels=64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.embedding(x)        # (batch_size, seq_len, embedding_dim)
        x = x.permute(0, 2, 1)       # (batch_size, embedding_dim, seq_len)
        x = self.conv1d(x)           # (batch_size, out_channels, seq_len)
        x = self.relu(x)
        x = self.global_avg_pool(x)  # (batch_size, out_channels, 1)
        x = x.squeeze(2)             # (batch_size, out_channels)
        x = self.fc(x)               # (batch_size, num_classes)
        return x

# Model instance
vocab_size = tokenizer.vocab_size
embedding_dim = 128  # You can choose the embedding dimension
num_classes = len(set(train_labels))
model = EmotionClassifier(vocab_size, embedding_dim, num_classes)

# ================ Training ===================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch, labels in train_loader:
        batch = {key: val.to(device) for key, val in batch.items()}
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(batch["input_ids"])
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss / len(train_loader):.4f}")

# ================ Evaluation ===================
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch, labels in test_loader:
        batch = {key: val.to(device) for key, val in batch.items()}
        labels = labels.to(device)

        outputs = model(batch["input_ids"])
        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total * 100
print(f"Test Accuracy: {accuracy:.2f}%")

# ================ Prediction ===================
def predict(sentence):
    model.eval()
    encoded = tokenize([sentence])
    encoded = {key: val.to(device) for key, val in encoded.items()}

    with torch.no_grad():
        output = model(encoded["input_ids"])
        prediction = torch.argmax(output, dim=1).cpu().item()

    emotions = ["sadness", "joy", "love", "anger", "fear", "surprise"]
    return emotions[prediction]

sample_sentence = "I’m so excited about this new adventure!"
print(f"Predicted Emotion: {predict(sample_sentence)}")

Encoding(num_tokens=87, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])